<a href="https://colab.research.google.com/github/pablobelmiro/olist_exploration/blob/main/06_previsao_demanda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce: Capítulo 6, Previsão de Demanda (+ bônus pro capítulo 3)

Continuação dos capítulos 1 a 5. Duas coisas nesse notebook: o capítulo 6 de verdade (previsão de pedidos por mês, o único ângulo de série temporal que o projeto ainda não tinha tocado), e uma exportação bônus pro capítulo 3, uma amostra geográfica com distância cliente-vendedor pra virar um mapa 3D (aproveitando que já vou recalcular a distância aqui mesmo). Não precisa de T4, só pandas e scikit-learn.

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

orders = pd.read_csv('olist_orders_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
print("dados carregados")

dados carregados


## A série mensal, e por que eu corto as pontas

O capítulo 1 já mostrou pedidos por mês, mas com 25 linhas incluindo pontas quase vazias (setembro/outubro/dezembro de 2016, com 4, 324 e 1 pedido, e setembro/outubro de 2018 com 16 e 4). Essas pontas não são sinal de demanda, são o dataset começando e terminando no meio de um mês. Pra previsão de verdade eu corto pra janeiro de 2017 até agosto de 2018, 20 meses cheios.

In [2]:
serie_completa = orders.groupby(orders['order_purchase_timestamp'].dt.to_period('M')).size()
serie = serie_completa['2017-01':'2018-08']

df = serie.reset_index()
df.columns = ['mes', 'pedidos']
df['t'] = np.arange(len(df))
df['mes_num'] = df['mes'].dt.month
df['sin'] = np.sin(2 * np.pi * df['mes_num'] / 12)
df['cos'] = np.cos(2 * np.pi * df['mes_num'] / 12)
print(f"{len(df)} meses, de {df['mes'].iloc[0]} a {df['mes'].iloc[-1]}")

20 meses, de 2017-01 a 2018-08


## Três candidatos, validados nos últimos 4 meses reais

Em vez de só extrapolar às cegas, separo os últimos 4 meses (maio a agosto de 2018) como teste e comparo três jeitos de prever: ingênuo (repete o último valor conhecido), sazonal ingênuo (repete o mesmo mês do ano anterior), e uma regressão linear com tendência mais sazonalidade cíclica (seno/cosseno do mês, não dummy por mês, que com só 20 pontos ia estourar em parâmetro).

In [3]:
n_holdout = 4
treino = df.iloc[:-n_holdout]
teste = df.iloc[-n_holdout:]

pred_ingenuo = np.array([treino['pedidos'].iloc[-1]] * n_holdout)

pred_sazonal = []
for _, row in teste.iterrows():
    mes_ano_anterior = row['mes'] - 12
    pred_sazonal.append(serie_completa.get(mes_ano_anterior, treino['pedidos'].mean()))
pred_sazonal = np.array(pred_sazonal)

modelo_linear = LinearRegression()
modelo_linear.fit(treino[['t', 'sin', 'cos']], treino['pedidos'])
pred_linear = modelo_linear.predict(teste[['t', 'sin', 'cos']])

y_real = teste['pedidos'].values
comparacao = []
for nome, pred in [
    ('Ingênuo (último mês)', pred_ingenuo),
    ('Sazonal ingênuo (mesmo mês, ano anterior)', pred_sazonal),
    ('Linear + sazonalidade cíclica', pred_linear),
]:
    mae = mean_absolute_error(y_real, pred)
    mape = mean_absolute_percentage_error(y_real, pred)
    comparacao.append({'model': nome, 'mae': round(float(mae), 1), 'mape': round(float(mape), 4)})
    print(f"{nome}: MAE={mae:.1f} MAPE={mape:.4f}")

with open('holdout-comparison.json', 'w', encoding='utf-8') as f:
    json.dump(comparacao, f, ensure_ascii=False, indent=2)

Ingênuo (último mês): MAE=478.0 MAPE=0.0758
Sazonal ingênuo (mesmo mês, ano anterior): MAE=2635.5 MAPE=0.4076
Linear + sazonalidade cíclica: MAE=2166.5 MAPE=0.3387


## O modelo mais simples ganhou, e isso é o achado do capítulo

O crescimento forte de 2017 (novembro é a Black Friday) fez o modelo linear extrapolar crescimento continuado pra maio-agosto de 2018, só que a base já tinha estabilizado, então ele erra disparado. O ingênuo, que só repete o último mês, ganha por larga margem. É o motivo pra usar ele (e não o mais sofisticado) na extrapolação final, com uma faixa de incerteza calculada a partir do erro real dele no teste, crescendo com a raiz do horizonte (3 meses à frente é mais incerto que 1 mês à frente).

In [4]:
residuos_holdout = y_real - pred_ingenuo
std_holdout = residuos_holdout.std()

ultimo_valor = int(df['pedidos'].iloc[-1])
futuro_meses = pd.period_range(df['mes'].iloc[-1] + 1, periods=3, freq='M')

previsao_futura = []
for i, mes in enumerate(futuro_meses, start=1):
    largura = std_holdout * np.sqrt(i) * 1.96
    previsao_futura.append({
        'month': str(mes),
        'predicted': ultimo_valor,
        'lower': round(max(0, ultimo_valor - largura), 1),
        'upper': round(ultimo_valor + largura, 1),
    })

historico = [{'month': str(row['mes']), 'orders': int(row['pedidos'])} for _, row in df.iterrows()]

resultado_previsao = {
    'history': historico,
    'holdout_start': str(teste['mes'].iloc[0]),
    'future': previsao_futura,
}
with open('demand-forecast.json', 'w', encoding='utf-8') as f:
    json.dump(resultado_previsao, f, ensure_ascii=False, indent=2)
print(json.dumps(previsao_futura, indent=2))

[
  {
    "month": "2018-09",
    "predicted": 6512,
    "lower": 5986.7,
    "upper": 7037.3
  },
  {
    "month": "2018-10",
    "predicted": 6512,
    "lower": 5769.1,
    "upper": 7254.9
  },
  {
    "month": "2018-11",
    "predicted": 6512,
    "lower": 5602.1,
    "upper": 7421.9
  }
]


## Bônus pro capítulo 3: amostra geográfica de distância

O capítulo 3 calculou `distance_km` (cliente-vendedor) mas nunca mostrou onde essa distância pesa mais no mapa. Recalculo aqui e exporto uma amostra de 500 pedidos com a localização do cliente e a distância, pra virar um scatter 3D (longitude/latitude no plano, distância na altura).

In [5]:
geo_media = geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

m = order_items.merge(orders[['order_id', 'customer_id']], on='order_id', how='left')
m = m.drop_duplicates(subset='order_id')
m = m.merge(customers[['customer_id', 'customer_zip_code_prefix']], on='customer_id', how='left')
m = m.merge(sellers[['seller_id', 'seller_zip_code_prefix']], on='seller_id', how='left')

m['customer_zip_code_prefix'] = m['customer_zip_code_prefix'].astype('int64')
m['seller_zip_code_prefix'] = m['seller_zip_code_prefix'].astype('int64')
geo_media['geolocation_zip_code_prefix'] = geo_media['geolocation_zip_code_prefix'].astype('int64')

m = m.merge(geo_media.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix', 'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'}), on='customer_zip_code_prefix', how='left')
m = m.merge(geo_media.rename(columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix', 'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'}), on='seller_zip_code_prefix', how='left')
m['distance_km'] = haversine(m['cust_lat'], m['cust_lng'], m['sell_lat'], m['sell_lng'])
m = m.dropna(subset=['cust_lat', 'cust_lng', 'distance_km'])

amostra = m.sample(n=min(500, len(m)), random_state=42)
amostra_json = [
    {
        'lat': round(float(r.cust_lat), 4),
        'lng': round(float(r.cust_lng), 4),
        'distance_km': round(float(r.distance_km), 1),
    }
    for r in amostra.itertuples()
]
with open('distance-3d-sample.json', 'w', encoding='utf-8') as f:
    json.dump(amostra_json, f, ensure_ascii=False, indent=2)
print(f"{len(amostra_json)} pedidos amostrados")

500 pedidos amostrados
